# AI Code Review Assistant Project

## Install Libraries

In [1]:
!pip install -q langchain langchain-community langchain-huggingface

!pip install transformers==4.52.4  langchain-classic

!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
moviepy 1.0.3 req

## Import Libraries

In [2]:

import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
import kagglehub
# Reading Jupyter Notebook
import json
#RAG
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
# langchain
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.language_models.llms import LLM
from typing import Any



/kaggle/input/datasets/do7amo7amed/projectcodereview/PythonCode.ipynb


/tmp/ipykernel_58/125587688.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## RAG

In [3]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_new_tokens=2048, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,      
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
!find /kaggle/input

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/do7amo7amed
/kaggle/input/datasets/do7amo7amed/projectcodereview
/kaggle/input/datasets/do7amo7amed/projectcodereview/PythonCode.ipynb


In [5]:
notebook_path = "/kaggle/input/datasets/do7amo7amed/projectcodereview/PythonCode.ipynb"
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

#Extract all Markdown
text = ""

for cell in notebook["cells"]:

    if cell["cell_type"] in ["markdown", "code"]:

        text += "".join(cell["source"])
        text += "\n\n"

In [6]:
print(text[:1000])

import json

students = []
def add_student(name, age, grade):
    student = {
        "name": name,
        "age": age,
        "grade": grade
    }
    students.append(student)
def load_students(filename):
    file = open(filename, "r")
    data = json.load(file)
    for s in data:
        students.append(s)
    file.close()
def save_students(filename):
    file = open(filename, "w")
    json.dump(students, file)
    file.close()
def find_student(name):
    for student in students:
        if student["name"] == name:
            return student
    return None
def calculate_average():
    total = 0
    for student in students:
        total += student["grade"]
    return total / len(students)
def delete_student(name):
    for student in students:
        if student["name"] == name:
            students.remove(student)
def print_students():
    for student in students:
        print(
            student["name"],
            student["age"],
            student["grade"]
        )
def upda

In [7]:
from langchain_core.documents import Document
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_text(text)
#Convert to LangChain Documents
documents = [
    Document(page_content=chunk)
    for chunk in chunks
]

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(documents, embedding)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
def retrieve_context(query):

    docs = vectordb.similarity_search(query, k=1)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    return context

In [9]:
def ask_question(query):
    docs = vectordb.similarity_search(query, k=1)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""
    
    result = generate_text(prompt, max_new_tokens=1024)
    return result.strip()
print(ask_question("What does this code do?"))

You are a helpful assistant. Use the following context to answer the question.

Context:
students = []
def add_student(name, age, grade):
    student = {
        "name": name,
        "age": age,
        "grade": grade
    }
    students.append(student)
def load_students(filename):
    file = open(filename, "r")
    data = json.load(file)
    for s in data:
        students.append(s)
    file.close()
def save_students(filename):
    file = open(filename, "w")
    json.dump(students, file)
    file.close()
def find_student(name):
    for student in students:
        if student["name"] == name:
            return student
    return None
def calculate_average():
    total = 0
    for student in students:
        total += student["grade"]
    return total / len(students)
def delete_student(name):
    for student in students:
        if student["name"] == name:
            students.remove(student)
def print_students():
    for student in students:
        print(
            student["name"],

### langchain

In [10]:
class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, max_new_tokens=2048)
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [11]:

analysis_prompt = PromptTemplate(
    input_variables=["code","context"],
    template="""
    You are a senior Python code reviewer.
    
    Use the documentation below as reference.
    
    Documentation:
    {context}
    
    Return ONLY:
    
    Summary:
    Potential Problems:
    Bugs:
    Bad Practices:
"""
)
analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

#-------------------------------------------------------
issue_prompt = PromptTemplate(
    input_variables=["analysis","format_instructions"],
    template="""
    You are a senior Python static code analyzer.

    Based on the analysis below:
    
    {analysis}
    
    Find REAL programming issues only.
    
    Examples:
    - Bugs
    - Runtime exceptions
    - Resource leaks
    - Bad coding practices
    - Security issues
    - Performance problems
    
    Ignore style preferences or subjective suggestions.

    Return ONLY one valid JSON object.
    
    {format_instructions}
"""
)

issue_chain = LLMChain(llm=llm,prompt=issue_prompt)
#----------------------------------------------------
improvement_prompt = PromptTemplate(
    input_variables=["code","issues"],
    template="""
    You are a senior Python developer.
    
    Using the following detected issues:
    
    {issues}
    
    Suggest improvements for this code.
    
    Code:
    
    {code}
    
    Return only:
    
    Issue:
    Suggested Fix:
    Improved Code:
"""
)

improvement_chain = LLMChain(llm=llm,prompt=improvement_prompt)

/tmp/ipykernel_58/2288509249.py:19: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)


## output parser

In [12]:
import re
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

issue_schema = [
     ResponseSchema(
        name="file",
        description="Name of the reviewed Python file"
    ),

    ResponseSchema(
        name="issue",
        description="Detected issue"
    ),

    ResponseSchema(
        name="severity",
        description="Low, Medium, or High"
    ),

    ResponseSchema(
        name="explanation",
        description="Why this is a problem"
    ),

    ResponseSchema(
        name="suggested_fix",
        description="How to fix the issue"
    )

]

output_parser = StructuredOutputParser.from_response_schemas(
    issue_schema
)

format_instructions = output_parser.get_format_instructions()

In [13]:
def extract_json_block(text):

    pattern = r"```json\s*(.*?)\s*```"

    matches = re.findall(pattern, text, re.DOTALL)

    if not matches:
        return None

    return matches[-1].strip()

In [14]:
def run_all(file_name, code):

    context = retrieve_context(code)

    analysis = analysis_chain.run({
        "code": code,
        "context": context
    })

    issues = issue_chain.run({
        "analysis": analysis,
        "format_instructions": format_instructions
    })

    improvements = improvement_chain.run({
        "code": code,
        "issues": issues
    })

    return analysis, issues, improvements

In [15]:
analysis, issues, improvements = run_all("PythonCodeReview.ipynb",text)

/tmp/ipykernel_58/4081796017.py:5: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  analysis = analysis_chain.run({


In [16]:
print(issues)


    You are a senior Python static code analyzer.

    Based on the analysis below:
    
    
    You are a senior Python code reviewer.
    
    Use the documentation below as reference.
    
    Documentation:
    students = []
def add_student(name, age, grade):
    student = {
        "name": name,
        "age": age,
        "grade": grade
    }
    students.append(student)
def load_students(filename):
    file = open(filename, "r")
    data = json.load(file)
    for s in data:
        students.append(s)
    file.close()
def save_students(filename):
    file = open(filename, "w")
    json.dump(students, file)
    file.close()
def find_student(name):
    for student in students:
        if student["name"] == name:
            return student
    return None
def calculate_average():
    total = 0
    for student in students:
        total += student["grade"]
    return total / len(students)
def delete_student(name):
    for student in students:
        if student["name"] == name:
   

In [17]:
json_text = extract_json_block(issues)

print(json_text)

{
	"file": "student_manager.py",
	"issue": "Missing input validation",
	"severity": "Medium",
	"explanation": "The code does not validate user input for name, age, and grade. This could lead to unexpected behavior or errors, especially if the input is not in the expected format. For example, an age or grade that is not a number could cause a runtime exception.",
	"suggested_fix": "Add input validation checks for name, age, and grade in all functions that accept these parameters. For example, age and grade could be validated using the int() and float() functions respectively. Name validation could be more complex and may require regular expressions or other techniques."
}


In [18]:
json_text = extract_json_block(issues)

parsed_issue = output_parser.parse(json_text)

print(parsed_issue)

{'file': 'student_manager.py', 'issue': 'Missing input validation', 'severity': 'Medium', 'explanation': 'The code does not validate user input for name, age, and grade. This could lead to unexpected behavior or errors, especially if the input is not in the expected format. For example, an age or grade that is not a number could cause a runtime exception.', 'suggested_fix': 'Add input validation checks for name, age, and grade in all functions that accept these parameters. For example, age and grade could be validated using the int() and float() functions respectively. Name validation could be more complex and may require regular expressions or other techniques.'}


In [19]:
print(json_text)

{
	"file": "student_manager.py",
	"issue": "Missing input validation",
	"severity": "Medium",
	"explanation": "The code does not validate user input for name, age, and grade. This could lead to unexpected behavior or errors, especially if the input is not in the expected format. For example, an age or grade that is not a number could cause a runtime exception.",
	"suggested_fix": "Add input validation checks for name, age, and grade in all functions that accept these parameters. For example, age and grade could be validated using the int() and float() functions respectively. Name validation could be more complex and may require regular expressions or other techniques."
}
